In [1]:
import os
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm import tqdm

# Paths — update to match your system
AUDIO_DIR    = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\FSC22\Audio Wise V1.0-20220916T202003Z-001\Audio Wise V1.0"
METADATA_CSV = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\FSC22\Metadata-20220916T202011Z-001\Metadata\Metadata V1.0 FSC22.csv"
OUTPUT_DIR   = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\processed"
CSV_OUTPUT   = r"C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\metadata.csv"

SAMPLE_RATE  = 44100
DURATION     = 5.0
TOP_DB       = 30

# Load and filter metadata
meta = pd.read_csv(METADATA_CSV)
print(f"Metadata loaded: {len(meta)} files")

Metadata loaded: 2025 files


In [2]:
def preprocess_audio(file_path, sr=SAMPLE_RATE, duration=DURATION):
    """
    Preprocesses a single audio file:
      1. Load as mono channel at 44100 Hz
      2. Trim silence from start and end (top_db=30)
      3. Pad with zeros if shorter than 5 seconds
      4. Cut to 5 seconds if longer

    Returns:
      audio (np.array): processed audio signal
      sr    (int)     : sample rate (44100)
    """
    try:
        # Step 1: Load audio — mono, resample to 44100 Hz
        audio, _ = librosa.load(file_path, sr=sr, mono=True)

        # Step 2: Trim silence from edges
        audio, _ = librosa.effects.trim(audio, top_db=TOP_DB)

        # Step 3: Fix to exactly 5 seconds
        target_length = int(sr * duration)
        if len(audio) < target_length:
            audio = np.pad(audio, (0, target_length - len(audio)))
        else:
            audio = audio[:target_length]

        return audio, sr

    except Exception as e:
        print(f"\n  [ERROR] {os.path.basename(file_path)}: {e}")
        return None, None

print("preprocess_audio() function defined successfully!")
print(f"  Target sample rate : {SAMPLE_RATE} Hz")
print(f"  Target duration    : {DURATION} seconds")
print(f"  Target length      : {int(SAMPLE_RATE * DURATION)} samples")
print(f"  Silence threshold  : {TOP_DB} dB")

preprocess_audio() function defined successfully!
  Target sample rate : 44100 Hz
  Target duration    : 5.0 seconds
  Target length      : 220500 samples
  Silence threshold  : 30 dB


In [3]:
THREAT_CLASSES = [
    "Fire", "Helicopter", "VehicleEngine", "Axe", "Chainsaw",
    "Generator", "Handsaw", "Firework", "Gunshot", "WoodChop", "TreeFalling"
]
def get_binary_label(class_name):
    return "threat" if class_name in THREAT_CLASSES else "wildlife"

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

records = []

print("Step 2: Preprocessing audio files...")

for _, row in tqdm(meta.iterrows(), total=len(meta)):
    filename     = row["Dataset File Name"]
    class_name   = row["Class Name"]
    binary_label = get_binary_label(class_name)
    file_path    = os.path.join(AUDIO_DIR, filename)

    audio, sr = preprocess_audio(file_path)

    if audio is not None:
        # Save into threat/ or wildlife/ subfolder
        out_dir  = os.path.join(OUTPUT_DIR, binary_label)
        os.makedirs(out_dir, exist_ok=True)

        out_name = f"{class_name}_{filename}"
        out_path = os.path.join(out_dir, out_name)
        sf.write(out_path, audio, sr)

        records.append({
            "filename"      : out_name,
            "original_class": class_name,
            "label"         : binary_label,
            "label_encoded" : 1 if binary_label == "threat" else 0,
            "sample_rate"   : sr,
            "duration_sec"  : DURATION,
            "processed_path": out_path
        })

print(f"\nPreprocessing complete!")
print(f"Total files saved: {len(records)}")
print(f"Output location  : {OUTPUT_DIR}")
print("Passing records to Nithin for metadata CSV generation")

Step 2: Preprocessing audio files...


  0%|          | 0/2025 [00:00<?, ?it/s]c:\ProgramData\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
100%|██████████| 2025/2025 [00:39<00:00, 50.81it/s]


Preprocessing complete!
Total files saved: 2025
Output location  : C:\Users\kanis\Downloads\IIMSTC\2nd Mini Project\data\processed
Passing records to Nithin for metadata CSV generation
